# B0: model gốc (Qwen2.5-VL-3B) trên 4 bộ của thiết kế 2 × 2, tập dev

Ảnh nguyên, không crop, không tiền xử lý; cùng cấu hình model như các nhánh crop (LLM fp16, vision tower fp32, eager,
`max_pixels = 1024·28·28`)
và cùng prompt chung (bản tiếng Anh cho TextVQA, DocVQA). Lưu kết quả từng câu vào `results/B0_base/`.

- TextVQA, DocVQA, ViTextVQA: 300 câu dev mỗi bộ (`data/qa/public/`; 100 câu ban đầu `dev_batch = "orig"`, 200 câu
  thêm ngày 18/11/2026 `"ext1"`). Chạy lại cả 100 câu cũ: cùng phiên bản thư viện thì kết quả phải trùng lượt trước.
- VisR-Bench tiếng Việt: phần dev của bộ nhãn cuối `data/qa/visr_bench_vi_v2.jsonl` (chia theo tài liệu bằng
  `src/data_prep/split_visr.py`). Mỗi câu ghi kèm `question_type`, `has_diacritic`, `target_size`, `eff_res`,
  `evidence_type` để tách nhóm khi báo cáo.

Chỉ chạy **dev**. Tập test để dành chạy một lần sau khi chốt cấu hình.

**Cần:** Kaggle dataset `da2-eval` (upload file `data/processed/kaggle/da2-eval.zip`, tạo bởi
`python -m src.data_prep.pack_kaggle --name da2-eval --datasets textvqa docvqa vitextvqa visr_bench_vi --splits dev`), GPU T4 x2,
Internet On.

**Kết quả:** tải `/kaggle/working/b0_dev_results.zip`, giải nén vào gốc repo.

In [ ]:
# Ghim phiên bản để các lượt chạy ở những ngày khác nhau (và các nhánh khác nhau) dùng cùng thư viện.
# transformers 5.20.2 là bản của lượt B0 đầu tiên với vision fp32 (18/11/2026).
!pip install -q transformers==5.20.2 accelerate qwen-vl-utils pillow numpy
import os, sys, shutil, glob

hits = glob.glob('/kaggle/input/**/src/vlm/qwen.py', recursive=True)
assert hits, 'Không thấy src/vlm/qwen.py: kiểm tra đã Add Input dataset da2-eval chưa'
# Có thể gắn nhiều dataset cùng chứa src/ (da2-havc, bản da2-eval cũ): chọn gốc có đủ file câu hỏi của lượt này
NEED = ['data/qa/visr_bench_vi_v2.jsonl']
roots = [os.path.dirname(os.path.dirname(os.path.dirname(h))) for h in hits]
ok = [r for r in roots if all(os.path.exists(os.path.join(r, f)) for f in NEED)]
assert ok, (f'Không gốc nào có {NEED}. Các gốc thấy: {roots}. '
            'Dataset da2-eval đang gắn là bản cũ: upload bản mới (New Version), rồi ở Input chọn version mới nhất.')
src_root = ok[0]
print('Dataset root:', src_root)
shutil.copytree(src_root, '/kaggle/working/da2', dirs_exist_ok=True)
os.chdir('/kaggle/working/da2'); sys.path.insert(0, '.')

import torch
TORCH_EXPECTED = '2.11.0'  # torch có sẵn trên Kaggle ở lượt B0 18/11/2026; không cài lại được dễ dàng nên chỉ kiểm
if not torch.__version__.startswith(TORCH_EXPECTED):
    print(f'CẢNH BÁO: torch {torch.__version__} khác {TORCH_EXPECTED} của các lượt trước; ghi lại và báo trước khi so nhánh')
print(torch.cuda.device_count(), 'GPU:', torch.cuda.get_device_name(0))

In [ ]:
import json, time, random
import numpy as np
from pathlib import Path
from PIL import Image
from src.vlm.qwen import load_qwen, answer, is_degenerate, DEFAULT_MODEL, DEFAULT_MAX_PIXELS
from src.vlm.prompts import answer_suffix, MAX_NEW_TOKENS, PROMPT_VERSION
from src.data_prep import eval_sets

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# VisR-Bench dev đã chạy ngày 18/11/2026 (results/B0_base/dev_visr_bench_vi_config.json); lượt này ba bộ công khai, 300 câu dev mỗi bộ.
# Chạy lại từ đầu thì dùng cả 4: [('textvqa', 'dev'), ('docvqa', 'dev'), ('vitextvqa', 'dev'), ('visr_bench_vi', 'dev')]
RUNS = [('textvqa', 'dev'), ('docvqa', 'dev'), ('vitextvqa', 'dev')]
CFG = dict(branch='B0', model=DEFAULT_MODEL, dtype='float16', vision_dtype='float32', attn='eager', max_pixels=DEFAULT_MAX_PIXELS,
           prompt_version=PROMPT_VERSION, max_new_tokens=MAX_NEW_TOKENS, decoding='greedy', seed=SEED,
           runs=RUNS, suffix_vi=answer_suffix('vitextvqa'), suffix_en=answer_suffix('textvqa'))
model, processor = load_qwen(CFG['model'], CFG['max_pixels'], CFG['attn'], vision_fp32=True)
import transformers, accelerate
from importlib.metadata import version as _ver
CFG.update(torch=torch.__version__, transformers=transformers.__version__, accelerate=accelerate.__version__,
           qwen_vl_utils=_ver('qwen-vl-utils'),
           gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

out_dir = Path('results/B0_base'); out_dir.mkdir(parents=True, exist_ok=True)
# Mỗi lượt một file cấu hình, đặt tên theo các bộ đã chạy, để lượt sau không ghi đè lượt trước
cfg_name = 'dev_' + '_'.join(d for d, _ in RUNS) + '_config.json'   # đuôi _config.json: .gitignore cho commit
json.dump(CFG, open(out_dir / cfg_name, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)

NGPU = torch.cuda.device_count()
def reset_peak():
    for g in range(NGPU):
        torch.cuda.reset_peak_memory_stats(g)
def peak_gb():
    # model chia trên các GPU (device_map='auto'): ghi VRAM đỉnh từng GPU, GiB
    return [round(torch.cuda.max_memory_allocated(g) / 2**30, 2) for g in range(NGPU)]

In [ ]:
for dataset, split in RUNS:
    items = eval_sets.load(dataset, split)
    suffix = answer_suffix(dataset)
    path = out_dir / f'{split}_{dataset}.jsonl'
    done = {json.loads(l)['qid'] for l in open(path, encoding='utf-8')} if path.exists() else set()
    with open(path, 'a', encoding='utf-8') as f:
        for i, it in enumerate(items):
            if it['qid'] in done:
                continue
            img = Image.open(it['image']).convert('RGB')
            reset_peak(); t0 = time.time()
            pred = answer(model, processor, img, it['question'], suffix, CFG['max_new_tokens'])
            rec = {k: it[k] for k in ('qid', 'dataset', 'document_id', 'question', 'gold', 'answers', 'answer_form', 'split')}
            rec.update({k: it[k] for k in ('dev_batch',) if it.get(k)})
            rec.update({k: it[k] for k in ('question_type', 'has_diacritic', 'target_size', 'eff_res', 'evidence_type') if k in it})
            rec.update(pred=pred, degenerate=is_degenerate(pred), image_size=list(img.size), sec=round(time.time() - t0, 2),
                       vram_gb=peak_gb())
            f.write(json.dumps(rec, ensure_ascii=False) + '\n'); f.flush()
            if i < 3 or i % 25 == 0:
                print(dataset, i, '|', it['question'][:60], '->', pred[:60], '| gold:', it['gold'][:40], f"| {rec['sec']}s")
    print(dataset, 'xong:', path)

In [ ]:
# Tóm tắt nhanh (chấm lấy điểm cao nhất trên các đáp án chấp nhận). Kiểm định ghép cặp chạy ở máy local bằng src/eval/stats.py
from src.eval.score import exact_match, f1, strip_diacritics
for dataset, split in RUNS:
    rows = [json.loads(l) for l in open(out_dir / f'{split}_{dataset}.jsonl', encoding='utf-8')]
    em = np.mean([max(exact_match(r['pred'], g) for g in r['answers']) for r in rows])
    ff = np.mean([max(f1(r['pred'], g) for g in r['answers']) for r in rows])
    ff_nd = np.mean([max(f1(strip_diacritics(r['pred']), strip_diacritics(g)) for g in r['answers']) for r in rows])
    sec = np.mean([r['sec'] for r in rows]); vram = np.max([r['vram_gb'] for r in rows], axis=0).tolist()
    deg = sum(r['degenerate'] for r in rows)
    print(f'{dataset:14s} {split:5s} n={len(rows):3d}  EM={em:.3f}  F1={ff:.3f}  F1_bỏ_dấu={ff_nd:.3f}  {sec:.1f}s/câu  VRAM đỉnh {vram} GiB  suy biến {deg}')

shutil.make_archive('/kaggle/working/b0_dev_results', 'zip', '.', 'results/B0_base')
print('Tải /kaggle/working/b0_dev_results.zip')